In [ ]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image
import albumentations as A
import cv2
import random

In [ ]:
# Define paths for the source IP102 dataset and target YOLO output directory
IP102_ROOT   = "ml_components/pest-detection/VOC2007"
ANNOTATIONS  = os.path.join(IP102_ROOT, "Annotations")
IMAGES       = os.path.join(IP102_ROOT, "JPEGImages")
IMAGESETS    = os.path.join(IP102_ROOT, "ImageSets/Main")
OUTPUT_DIR   = "ml_components/pest-detection/cinnamon_pests_yolo"

# Configuration variables for data splitting and reproducibility
VAL_SPLIT    = 0.2
RANDOM_SEED  = 42

# Define the target 8-class schema for cinnamon pests
CINNAMON_PEST_CLASSES = {
    0: "stem_borer",
    1: "thrips",
    2: "moth",
    3: "mite",
    4: "leaf_miner",
    5: "root_grub",
    6: "caterpillar",
    7: "weevil",
}

# Map specific numeric IP102 class IDs to the target 8-class schema
NUMERIC_MAP = {
    # stem_borer
    "4": 0, "5": 0, "23": 0, "27": 0,
    "66": 0, "69": 0, "100": 0,
    # thrips
    "13": 1, "34": 1, "54": 1, "55": 1, "93": 1,
    # moth
    "18": 2, "42": 2, "46": 2, "59": 2, "87": 2, "98": 2,
    # mite
    "22": 3, "32": 3, "33": 3, "61": 3,
    "62": 3, "64": 3, "75": 3, "76": 3,
    # leaf_miner
    "36": 4, "37": 4, "89": 4,
    # root_grub
    "15": 5, "44": 5,
    # caterpillar
    "1": 6, "2": 6, "19": 6, "20": 6,
    "21": 6, "24": 6, "39": 6, "40": 6,
    # weevil
    "11": 7, "43": 7, "45": 7, "97": 7, "101": 7,
}

In [ ]:
# Map specific string IP102 class names to the target 8-class schema
NAME_MAP = {
    # stem_borer
    "asiatic rice borer": 0, "yellow rice borer": 0,
    "corn borer": 0, "peach borer": 0,
    "parathrene regalis": 0, "xylotrechus": 0,
    "rhytidodera bowrinii white": 0,
    # thrips
    "grain spreader thrips": 1, "wheat phloeothrips": 1,
    "odontothrips loti": 1, "thrips": 1,
    "scirtothrips dorsalis hood": 1,
    # moth
    "white margined moth": 2, "meadow moth": 2,
    "flax budworm": 2, "limacodidae": 2,
    "prodenia litura": 2, "chlumetia transversa": 2,
    # mite
    "red spider": 3, "penthaleus major": 3,
    "longlegged spider mite": 3, "colomerus vitis": 3,
    "brevipoalpus lewisi mcgregor": 3,
    "polyphagotars onemus latus": 3,
    "panonchus citri mcgregor": 3,
    "phyllocoptes oleiverus ashmead": 3,
    # leaf_miner
    "cerodonta denticornis": 4, "beet fly": 4,
    "phyllocnistis citrella stainton": 4,
    # root_grub
    "grub": 5, "sericaorient alismots chulsky": 5,
    # caterpillar
    "rice leaf roller": 6, "rice leaf caterpillar": 6,
    "black cutworm": 6, "large cutworm": 6,
    "yellow cutworm": 6, "army worm": 6,
    "cabbage army worm": 6, "beet army worm": 6,
    # weevil
    "rice water weevil": 7, "beet weevil": 7,
    "alfalfa weevil": 7, "deporaus marginatus pascoe": 7,
    "sternochetus frigidus": 7,
}

In [ ]:
def resolve_class(name_or_id: str):
    name_or_id = name_or_id.strip()
    # Try numeric first
    if name_or_id in NUMERIC_MAP:
        return NUMERIC_MAP[name_or_id]
    # Try name
    if name_or_id.lower() in NAME_MAP:
        return NAME_MAP[name_or_id.lower()]
    return None

def parse_xml(xml_path):
    try:
        # Load and parse the XML annotation file
        tree = ET.parse(xml_path)
    except ET.ParseError as e:
        # Catch the corrupted XML, print a warning, and skip this file
        print(f"\nSkipping corrupted XML file: {xml_path}")
        print(f"   Reason: {e}")
        return []

    root = tree.getroot()

    size   = root.find("size")
    if size is None:
        return []

    # Extract image dimensions required for YOLO normalization
    img_w_elem = size.find("width")
    img_h_elem = size.find("height")
    
    # Catch cases where width/height might be missing entirely
    if img_w_elem is None or img_h_elem is None:
        return []
        
    img_w = int(img_w_elem.text)
    img_h = int(img_h_elem.text)

    # Prevent division by zero later
    if img_w == 0 or img_h == 0:
        return []  # corrupt annotation

    yolo_rows = []
    # Iterate through all detected objects in the XML
    for obj in root.findall("object"):
        raw_name     = obj.find("name").text.strip()
        cinnamon_cls = resolve_class(raw_name)

        if cinnamon_cls is None:
            continue   # not a cinnamon-relevant pest

        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        # Clamp 
        xmin = max(0, min(xmin, img_w))
        xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h))
        ymax = max(0, min(ymax, img_h))

        # Convert bounding box to YOLO format
        x_center = (xmin + xmax) / (2 * img_w)
        y_center = (ymin + ymax) / (2 * img_h)
        width    = (xmax - xmin) / img_w
        height   = (ymax - ymin) / img_h

        if width <= 0 or height <= 0:
            continue

        # Append formatted string to rows list
        yolo_rows.append(
            f"{cinnamon_cls} {x_center:.6f} {y_center:.6f} "
            f"{width:.6f} {height:.6f}"
        )

    return yolo_rows

In [ ]:
def read_image_ids(txt_path):
    # Read non-empty lines from the provided text file
    with open(txt_path) as f:
        return [line.strip() for line in f if line.strip()]


def write_split(image_ids, split_name, class_counts):
    # Setup target directories for images and labels
    out_img = Path(OUTPUT_DIR) / split_name / "images"
    out_lbl = Path(OUTPUT_DIR) / split_name / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    written  = 0
    skipped  = 0 

    # Process each image ID assigned to this split
    for img_id in image_ids:
        xml_path = os.path.join(ANNOTATIONS, f"{img_id}.xml")
        
        # Find the image
        img_path = None
        for ext in [".jpg", ".jpeg", ".JPG", ".png"]:
            candidate = os.path.join(IMAGES, f"{img_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break

        # Skip if annotation or image file is missing
        if not os.path.exists(xml_path) or img_path is None:
            skipped += 1
            continue

        # Extract YOLO formatted bounding boxes
        yolo_rows = parse_xml(xml_path)

        if not yolo_rows:
            skipped += 1
            continue   # image has no cinnamon-relevant pests

        # Count per class to track distribution
        for row in yolo_rows:
            cls_id = int(row.split()[0])
            class_counts[split_name][cls_id] += 1

        # Copy image to the appropriate output folder
        dst_img = out_img / Path(img_path).name
        shutil.copy2(img_path, dst_img)

        # Write label file to the appropriate output folder
        lbl_path = out_lbl / f"{img_id}.txt"
        lbl_path.write_text("\n".join(yolo_rows))
        written += 1

    return written, skipped

In [ ]:
def run_conversion():
    # Set seed for reproducible random splits
    random.seed(RANDOM_SEED)
    class_counts = {
        "train": {i: 0 for i in range(9)},
        "val":   {i: 0 for i in range(9)},
        "test":  {i: 0 for i in range(9)},
    }

    # Split trainval
    trainval_ids = read_image_ids(os.path.join(IMAGESETS, "trainval.txt"))
    random.shuffle(trainval_ids)
    
    # Calculate index to split training and validation data
    val_size   = int(len(trainval_ids) * VAL_SPLIT)
    val_ids    = trainval_ids[:val_size]
    train_ids  = trainval_ids[val_size:]
    test_ids   = read_image_ids(os.path.join(IMAGESETS, "test.txt"))

    splits = [("train", train_ids), ("val", val_ids), ("test", test_ids)]

    # Execute file parsing and copying for each subset
    for split_name, ids in splits:
        written, skipped = write_split(ids, split_name, class_counts)
        print(f"\n── {split_name.upper()} ──")
        print(f"  Written : {written}")
        print(f"  Skipped : {skipped} (no relevant annotations)")
        for cls_id, cls_name in CINNAMON_PEST_CLASSES.items():
            print(f"  {cls_name:20s}: {class_counts[split_name][cls_id]:>4} boxes")

    # Generate the dataset YAML file
    write_yaml()
    print("\nConversion complete.")

    def write_yaml():
        # Construct YOLO configuration file content
        yaml_content = f"""path: {OUTPUT_DIR}

train: train/images
val:   val/images
test:  test/images

nc: 9

names:
  0: stem_borer
  1: thrips
  2: moth
  3: mite
  4: leaf_miner
  5: root_grub
  6: caterpillar
  7: weevil
"""
    yaml_path = os.path.join(OUTPUT_DIR, "cinnamon_pests.yaml")
    Path(yaml_path).write_text(yaml_content)
    print(f"\nYAML written to {yaml_path}")


# Execute conversion script
if __name__ == "__main__":
    run_conversion()

In [ ]:
def verify_dataset():
    # Loop through each dataset split to verify file alignment and format
    for split in ["train", "val", "test"]:
        lbl_dir  = Path(OUTPUT_DIR) / split / "labels"
        img_dir  = Path(OUTPUT_DIR) / split / "images"
        
        labels   = list(lbl_dir.glob("*.txt"))
        images   = list(img_dir.glob("*"))
        
        print(f"\n{split}: {len(images)} images, {len(labels)} label files")
        
        # Every image must have a label and vice versa
        label_stems = {f.stem for f in labels}
        image_stems = {f.stem for f in images}
        
        # Identify missing or unlinked files
        missing_labels = image_stems - label_stems
        missing_images = label_stems - image_stems
        
        if missing_labels:
            print(f"  WARNING: {len(missing_labels)} images with no label")
        if missing_images:
            print(f"  WARNING: {len(missing_images)} labels with no image")
        
        # Spot-check label file format for errors
        errors = 0
        for lbl in labels:
            for line in lbl.read_text().strip().split("\n"):
                parts = line.split()
                if len(parts) != 5:
                    errors += 1
                    continue
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                if cls not in range(9):
                    errors += 1
                if any(c < 0 or c > 1 for c in coords):
                    errors += 1
        
        print(f"  Format errors: {errors}")

# Run initial verification
verify_dataset()

In [ ]:
# Per-class oversample factor 
OVERSAMPLE_CONFIG = {
    1: {"name": "thrips",      "factor": 4},
    4: {"name": "leaf_miner",  "factor": 4},
    2: {"name": "moth",        "factor": 2},
    0: {"name": "stem_borer",  "factor": 2},
    3: {"name": "mite",       "factor": 2},
    5: {"name": "root_grub",  "factor": 2},
}

# Define the data augmentation pipeline
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.Rotate(limit=20, p=0.5),
    A.GaussNoise(p=0.2),
    A.HueSaturationValue(p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


def safe_class_id(token: str) -> int:
    return int(float(token))


def clean_previous_aug_files(split):
    img_dir = Path(OUTPUT_DIR) / split / "images"
    lbl_dir = Path(OUTPUT_DIR) / split / "labels"

    # Search for and delete existing augmented files
    removed = 0
    for f in list(img_dir.glob("*_aug*.jpg")) + list(lbl_dir.glob("*_aug*.txt")):
        f.unlink()
        removed += 1

    print(f"Cleaned {removed} leftover augmented files from {split}")


def oversample_class(split, class_id, factor, already_augmented):

    img_dir = Path(OUTPUT_DIR) / split / "images"
    lbl_dir = Path(OUTPUT_DIR) / split / "labels"

    # Identify files that need to be augmented
    target_files = []
    for lbl_file in lbl_dir.glob("*.txt"):
        if "_aug" in lbl_file.stem:
            continue
        if lbl_file.stem in already_augmented:
            continue  # already oversampled for a different target class this run

        lines = lbl_file.read_text().strip().split("\n")
        if not lines or lines == ['']:
            continue

        # Check if the file contains the target class ID
        try:
            has_class = any(
                safe_class_id(line.split()[0]) == class_id
                for line in lines
            )
        except (ValueError, IndexError):
            print(f"  Skipping malformed label file: {lbl_file.name}")
            continue

        if has_class:
            target_files.append(lbl_file)

    print(f"Oversampling {len(target_files)} images for class {class_id} (factor={factor})")

    # Apply data augmentation to selected target files
    for lbl_file in target_files:
        img_file = img_dir / (lbl_file.stem + ".jpg")
        if not img_file.exists():
            continue

        image = cv2.imread(str(img_file))
        if image is None:
            print(f"  Could not read image: {img_file.name}")
            continue

        lines = lbl_file.read_text().strip().split("\n")

        # Extract bounding boxes and class labels from file
        bboxes = []
        class_labels = []
        for line in lines:
            parts = line.split()
            class_labels.append(safe_class_id(parts[0]))
            bboxes.append([float(x) for x in parts[1:]])

        # Generate copies based on the designated oversample factor
        for i in range(factor):
            try:
                augmented = augment(image=image, bboxes=bboxes, class_labels=class_labels)
            except Exception as e:
                print(f"Augmentation failed for {lbl_file.name} (attempt {i}): {e}")
                continue

            if not augmented['bboxes']:
                continue  # all boxes got dropped by augmentation

            # Define new filenames for augmented outputs
            new_img_name = f"{lbl_file.stem}_aug{i}.jpg"
            new_lbl_name = f"{lbl_file.stem}_aug{i}.txt"

            # Save the new augmented image
            cv2.imwrite(str(img_dir / new_img_name), augmented['image'])

            # Format and save the new bounding box coordinates
            new_lines = []
            for cls, box in zip(augmented['class_labels'], augmented['bboxes']):
                cls_int = int(cls)  # force int, never float, when writing
                new_lines.append(f"{cls_int} {' '.join(f'{c:.6f}' for c in box)}")
            (lbl_dir / new_lbl_name).write_text("\n".join(new_lines))

        # Add file stem to tracker to prevent duplicate multi-class augmentations
        already_augmented.add(lbl_file.stem)

        clean_previous_aug_files("train")

already_augmented = set()  # tracks stems already processed, across all classes

# Process heavier-oversample classes first so they get priority
for class_id, cfg in sorted(OVERSAMPLE_CONFIG.items(), key=lambda x: -x[1]["factor"]):
    oversample_class("train", class_id, cfg["factor"], already_augmented)

print("\nOversampling complete. Run recount_classes('train') to verify new distribution.")

In [ ]:
def visualize_sample(split="train", n=10):
    img_dir = Path(OUTPUT_DIR) / split / "images"
    lbl_dir = Path(OUTPUT_DIR) / split / "labels"
    
    # Grab a random subset of label files
    label_files = list(lbl_dir.glob("*.txt"))
    sample = random.sample(label_files, min(n, len(label_files)))

    # Create directory for outputting visualization samples
    out_dir = Path(OUTPUT_DIR) / "sanity_check"
    out_dir.mkdir(exist_ok=True)

    names = ["stem_borer", "thrips", "moth", "mite", "leaf_miner",
             "root_grub", "caterpillar", "weevil"]

    # Loop through the random sample, draw bounding boxes, and render text labels
    for lbl_file in sample:
        img_file = img_dir / (lbl_file.stem + ".jpg")
        if not img_file.exists():
            continue
        img = cv2.imread(str(img_file))
        h, w = img.shape[:2]

        for line in lbl_file.read_text().strip().split("\n"):
            cls, xc, yc, bw, bh = line.split()
            cls = int(cls)
            # Denormalize YOLO coordinates to raw pixels
            xc, yc, bw, bh = float(xc)*w, float(yc)*h, float(bw)*w, float(bh)*h
            x1, y1 = int(xc - bw/2), int(yc - bh/2)
            x2, y2 = int(xc + bw/2), int(yc + bh/2)
            # Draw rectangle and add class label text
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, names[cls], (x1, max(y1-5, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Save the labeled image to disk
        cv2.imwrite(str(out_dir / img_file.name), img)

    print(f"Saved {len(sample)} sanity-check images to {out_dir}")

# Generate visualizations
visualize_sample("train", n=15)

In [ ]:
def verify_dataset():
    # Loop through datasets to identify missing files or malformed boxes post-augmentation
    for split in ["train", "val", "test"]:
        lbl_dir  = Path(OUTPUT_DIR) / split / "labels"
        img_dir  = Path(OUTPUT_DIR) / split / "images"
        
        labels   = list(lbl_dir.glob("*.txt"))
        images   = list(img_dir.glob("*"))
        
        print(f"\n{split}: {len(images)} images, {len(labels)} label files")
        
        # Every image must have a label and vice versa
        label_stems = {f.stem for f in labels}
        image_stems = {f.stem for f in images}
        
        missing_labels = image_stems - label_stems
        missing_images = label_stems - image_stems
        
        if missing_labels:
            print(f"  WARNING: {len(missing_labels)} images with no label")
        if missing_images:
            print(f"  WARNING: {len(missing_images)} labels with no image")
        
        # Spot-check label file format
        errors = 0
        for lbl in labels:
            for line in lbl.read_text().strip().split("\n"):
                parts = line.split()
                if len(parts) != 5:
                    errors += 1
                    continue
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                if cls not in range(9):
                    errors += 1
                if any(c < 0 or c > 1 for c in coords):
                    errors += 1
        
        print(f"  Format errors: {errors}")

# Run post-augmentation verification
verify_dataset()

def recount_classes(split="train"):
    # Calculate the new distribution of classes in the given split
    lbl_dir = Path(OUTPUT_DIR) / split / "labels"
    counts = {i: 0 for i in range(8)}

    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().split("\n"):
            if not line:
                continue
            cls = int(float(line.split()[0]))
            counts[cls] += 1

    names = ["stem_borer", "thrips", "moth", "mite", "leaf_miner",
              "root_grub", "caterpillar", "weevil"]
    print(f"\n{split} class distribution after oversampling:")
    for i, name in enumerate(names):
        print(f"  {name:15s}: {counts[i]:>5}")

# Display final class totals
recount_classes("train")